# Cumulative Incidence Curves: 6 Toxicities (Figure 2A)

Kaplan-Meier cumulative incidence curves for 6 irAE types. 

In [ ]:
%matplotlib inline
import re, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 450,
})
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from lifelines import KaplanMeierFitter

def standardize_mrn(mrn):
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r"\d+", str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None

# Constants
TOXICITY_COLUMNS = ["pneumonitis", "adrenal_insufficiency", "liver_toxicity",
                    "colitis", "hyperthyroidism", "hypothyroidism"]
TOXICITY_DISPLAY = {
    "pneumonitis": "Pneumonitis", "adrenal_insufficiency": "Adrenal Insufficiency",
    "liver_toxicity": "Liver Toxicity", "colitis": "Colitis",
    "hyperthyroidism": "Hyperthyroidism", "hypothyroidism": "Hypothyroidism",
}
AE_COLORS = {
    "pneumonitis": "#D55E00", "adrenal_insufficiency": "#009E73",
    "liver_toxicity": "#0072B2", "colitis": "#E69F00",
    "hyperthyroidism": "#56B4E9", "hypothyroidism": "#CC79A7",
}

In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
TOX_TABLE_DIR = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'main'))
os.makedirs(RESULTS_DIR, exist_ok=True)

BACKBONE_PATH = os.path.join(TOX_TABLE_DIR, 'llm84k_pneumonitis_grade0_20260630.csv')
covars = pd.read_csv(BACKBONE_PATH, low_memory=False)
batch_df = pd.read_csv(os.path.join(DATA_DIR, 'llm_calls_batch_level_84k.csv'), encoding="latin-1", low_memory=False)

# Standardize MRNs
covars["mrn"] = covars["mrn"].apply(standardize_mrn)
batch_df["mrn"] = batch_df["mrn"].apply(standardize_mrn)
covars = covars[covars["mrn"].notna()].copy()
batch_df = batch_df[batch_df["mrn"].notna()].copy()

# Parse dates
covars["lot_start"] = pd.to_datetime(covars["lot_start"], errors="coerce")
covars["lot"] = pd.to_numeric(covars["lot"], errors="coerce")
covars["censor_days"] = pd.to_numeric(covars["t_cutoff_lot"], errors="coerce")  # pre-computed, team-consistent
covars = covars.sort_values(["mrn", "lot"])

batch_df["window_start"] = pd.to_datetime(batch_df["window_start"], errors="coerce")
batch_df["window_end"] = pd.to_datetime(batch_df["window_end"], errors="coerce")
batch_df = batch_df.dropna(subset=["window_start", "window_end"])

print(f"Covars: {covars.shape}")
print(f"Batch: {batch_df.shape}, {batch_df['mrn'].nunique():,} patients")

In [ ]:
covars_valid = covars[covars["lot_start"].notna()].copy()
line1 = (covars_valid.sort_values(["mrn", "lot"])
         .groupby("mrn").first().reset_index()[["mrn", "lot_start", "censor_days"]]
         .rename(columns={"lot_start": "line1_start"}))
line1 = line1[np.isfinite(line1["censor_days"]) & (line1["censor_days"] > 0)].copy()
print(f"Line 1 patients with valid censoring: {len(line1):,}")

In [ ]:
# Merge batch with line1 and compute days from start
batch_merged = batch_df.merge(line1, on="mrn", how="inner")
batch_merged["days_from_start"] = (batch_merged["window_start"] - batch_merged["line1_start"]).dt.days
batch_merged = batch_merged[
    (batch_merged["days_from_start"] >= 0) &
    (batch_merged["days_from_start"] <= batch_merged["censor_days"])
].copy()
print(f"Batch records after merge & filtering: {len(batch_merged):,}")

In [ ]:
# Fit KM cumulative incidence for each toxicity
MAX_MONTHS = 24
tick_months = np.arange(0, MAX_MONTHS + 1, 3)

kmf_dict, surv_dict = {}, {}
for tox in TOXICITY_COLUMNS:
    if tox not in batch_merged.columns:
        continue
    ae_records = batch_merged[batch_merged[tox] == 1].copy()
    first_ae = ae_records.groupby("mrn")["days_from_start"].min().reset_index().rename(
        columns={"days_from_start": "time"})
    first_ae["event"] = 1
    surv = line1[["mrn", "censor_days"]].merge(first_ae, on="mrn", how="left")
    surv["event"] = surv["event"].fillna(0).astype(int)
    surv.loc[surv["event"] == 0, "time"] = surv.loc[surv["event"] == 0, "censor_days"]
    surv.loc[(surv["event"] == 1) & (surv["time"] > surv["censor_days"]), "event"] = 0
    surv.loc[surv["event"] == 0, "time"] = surv["censor_days"]
    surv = surv[surv["time"] > 0].copy()
    surv["time_months"] = surv["time"] / 30.44
    kmf = KaplanMeierFitter()
    kmf.fit(surv["time_months"], surv["event"], label=TOXICITY_DISPLAY[tox])
    kmf_dict[tox] = kmf
    surv_dict[tox] = surv
    print(f"  {TOXICITY_DISPLAY[tox]}: n={len(surv):,}, events={int(surv['event'].sum()):,}")

In [ ]:
# Compute y-axis limit
max_ci = 0
for tox in TOXICITY_COLUMNS:
    if tox not in kmf_dict:
        continue
    ci = 1 - kmf_dict[tox].survival_function_
    label = TOXICITY_DISPLAY[tox]
    mask = ci.index <= MAX_MONTHS
    max_ci = max(max_ci, ci[label][mask].max() * 100)
y_max = min(50, max(5, max_ci * 1.2))
y_max = np.ceil(y_max / 5) * 5
row_order = [tox for tox in TOXICITY_COLUMNS if tox in kmf_dict]
n_aes = len(row_order)
# Plot
fig, ax = plt.subplots(figsize=(4.2, 2.6))
for tox in row_order:
    kmf = kmf_dict[tox]
    surv = surv_dict[tox]
    label = TOXICITY_DISPLAY[tox]
    ci = 1 - kmf.survival_function_
    ci_vals = ci[label].values * 100
    ci_times = ci.index.values
    ax.plot(ci_times, ci_vals, linewidth=1.6, color=AE_COLORS[tox],
        label=f"{label} (n={len(surv):,}, events={int(surv['event'].sum()):,})")
    # Censoring ticks
    censored = surv[surv["event"] == 0]["time_months"].values
    censored = censored[(censored > 0) & (censored <= MAX_MONTHS)]
    if len(censored) > 200:
        np.random.seed(42)
        censored = np.random.choice(censored, 200, replace=False)
    tick_h = y_max * 0.012
    for ct in censored:
        idx = max(0, min(np.searchsorted(ci_times, ct, side="right") - 1, len(ci_vals) - 1))
        ax.plot([ct, ct], [ci_vals[idx] - tick_h, ci_vals[idx] + tick_h],
                color=AE_COLORS[tox], linewidth=0.8, alpha=0.7, solid_capstyle="butt")
ax.set_xlim(0, MAX_MONTHS)
ax.set_ylim(0, y_max)
ax.set_xticks(tick_months)
ax.set_xticklabels([str(int(m)) for m in tick_months], fontsize=6)
ax.set_xlabel("Months from Line 1 treatment start", fontsize=7)
ax.set_ylabel("Cumulative incidence (%)", fontsize=7)
ax.tick_params(axis="y", labelsize=6)
ax.legend(loc="upper left", fontsize=5, framealpha=0.9, handlelength=1.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.subplots_adjust(left=0.12, right=0.97, top=0.95, bottom=0.15)

In [ ]:
# Save figure
os.makedirs(RESULTS_DIR, exist_ok=True)
with PdfPages(os.path.join(RESULTS_DIR, 'Cumulative_Incidence_2A.pdf')) as pdf:
    pdf.savefig(fig, dpi=450)
plt.close(fig)
print("Saved: ../results/main/Cumulative_Incidence_2A.pdf")

In [ ]:
# Summary CSV: N and events per toxicity
summary_rows = []
for tox in row_order:
    surv = surv_dict[tox]
    summary_rows.append({
        "toxicity": TOXICITY_DISPLAY[tox],
        "n": len(surv),
        "events": int(surv["event"].sum()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(RESULTS_DIR, 'Cumulative_Incidence_2A_summary.csv'), index=False)
summary_df

In [ ]:
# Full at-risk table CSV (not shown in the plot anymore, but saved for reference)
risk_rows = []
for tox in row_order:
    surv = surv_dict[tox]
    row = {"toxicity": TOXICITY_DISPLAY[tox]}
    for t in tick_months:
        n_at_risk = int((surv["time_months"] >= t).sum())
        row[f"month_{int(t)}"] = n_at_risk
    risk_rows.append(row)
risk_df = pd.DataFrame(risk_rows)
risk_df.to_csv(os.path.join(RESULTS_DIR, 'Cumulative_Incidence_2A_at_risk_table.csv'), index=False)
risk_df